# CERDOS 55 · Mask R-CNN con tasa de aprendizaje decreciente (coseno) bajo el mismo presupuesto L4
### Análisis de sensibilidad del experimento 54 · FaroPigSeg

**Pregunta:** en el experimento 54, Mask R-CNN usó una tasa de aprendizaje **constante** (AdamW 1e-4), mientras que las recetas de YOLO la **reducen** al final del entrenamiento. ¿Cambia el resultado si Mask R-CNN también la reduce?

**Único cambio respecto al NB54:** la tasa de aprendizaje de Mask R-CNN baja con una curva coseno desde 1e-4 hasta 1e-6 (1 % del valor inicial, la misma fracción final `lrf=0.01` que usa Ultralytics), en función del **tiempo transcurrido** dentro del presupuesto de **60 min** en la misma NVIDIA L4. Todo lo demás es idéntico: particiones congeladas (verificadas por SHA-256), pesos COCO, congelamiento parcial del backbone, AdamW, batch 1, 512–768 px, validación COCO cada 2 épocas, selección del mejor checkpoint por AP de validación y semillas 2026, 2027 y 2028.

**El test no se evalúa en este notebook.**

**Cómo usarlo:** GPU **L4** → **Entorno de ejecución → Ejecutar todas**. Entrena las tres semillas seguidas (≈3 h 10 min) y salta las que ya terminaron. Si la sesión se corta, al volver a ejecutar continúa desde la última época guardada (incluida la posición en la curva de la tasa de aprendizaje).


## 1. Configuración (no hay que editar nada)

In [ ]:
BUDGET_MINUTES_PER_RUN = 60.0   # idéntico al NB54
SEEDS = (2026, 2027, 2028)
GPU_NAME_REQUIRED = 'L4'
MASK_BATCH = 1
MASK_EPOCH_CAP = 250
LR0 = 1e-4                      # igual que NB54
LR_FINAL_FRACTION = 0.01        # tasa final = 1e-6 (misma fracción final que lrf=0.01 de Ultralytics)
RELATIVE_WALL_TIME_TOLERANCE = 0.10
PROTOCOL_ID = 'CERDOS55_MASKRCNN_COSINE_L4_v1'
assert BUDGET_MINUTES_PER_RUN == 60.0
print('NB55 | Mask R-CNN con LR coseno', LR0, '->', LR0*LR_FINAL_FRACTION, '| 60 min por semilla | semillas', SEEDS)


## 2. Entorno, librerías y Drive

In [ ]:
import importlib.util, subprocess, sys
required = {'pycocotools':'pycocotools', 'yaml':'pyyaml', 'cv2':'opencv-python-headless', 'pandas':'pandas', 'PIL':'Pillow'}
missing=[pip for mod,pip in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import os, json, yaml, time, hashlib, platform, random, gc, shutil, io
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, torchvision, cv2, matplotlib.pyplot as plt
from google.colab import drive
assert torch.cuda.is_available(), 'Activar GPU en Colab antes de continuar.'
GPU = torch.cuda.get_device_name(0)
print('GPU:',GPU,'| Torch:',torch.__version__,'| Torchvision:',torchvision.__version__)
assert GPU_NAME_REQUIRED.lower() in GPU.lower(), f'Esta comparación estaba diseñada para L4: {GPU}'
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Carpeta original no localizada: {ROOT}'
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
YAML=SPLIT/'data_fecha.yaml'; MANIFEST=SPLIT/'particion_fecha_manifest.json'
OUT=ROOT/'03_Resultados'/'55_MaskRCNN_LR_Coseno_L4'
OUT54=ROOT/'03_Resultados'/'54_Presupuesto_Equiparado_L4'   # solo lectura, para comparar
OUT.mkdir(parents=True,exist_ok=True)
assert all(x.is_file() for x in (YAML,MANIFEST)), 'Faltan los manifiestos congelados; no reconstruir particiones.'
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def write_json_atomic(path, payload):
    p=Path(path); tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(payload,indent=2,ensure_ascii=False,default=str),encoding='utf-8');os.replace(tmp,p)
print('Salida nueva:',OUT)


## 3. Auditoría de particiones congeladas (sin evaluar el test)

In [ ]:
# SHA256 fijado por el experimento original NB32. FAIL-CLOSED ante cualquier variación.
EXPECTED_YAML_SHA='9a223b47038c661303c1d12a35cbdcb212ed51529c8c773c4c9c66998706d1d9'
assert sha256_file(YAML)==EXPECTED_YAML_SHA, 'El YAML difiere del estudio original: ABORTAR.'
cf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert set(manifest['asignaciones_fecha'].values())=={'train','val','test'}
parts={}; dates={}
for s,n_expected,d_expected in [('train',949,16),('val',196,4),('test',373,5)]:
    p=Path(cf[s]); assert p.is_file(),f'Falta lista {s}: {p}'
    parts[s]=[Path(x.strip()) for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
    dates[s]={p.name[:10] for p in parts[s]}
    assert len(parts[s])==n_expected and len(dates[s])==d_expected,(s,len(parts[s]),len(dates[s]))
    assert len({str(p) for p in parts[s]})==len(parts[s])
    assert all(manifest['asignaciones_fecha'].get(d)==s for d in dates[s]), 'Fechas contra manifiesto no coinciden'
    print(s,':',len(parts[s]),'imágenes,',len(dates[s]),'fechas')
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not ({str(p) for p in parts[a]}&{str(p) for p in parts[b]})
    assert not(dates[a]&dates[b])
# Sólo train y val se inspeccionan o cargan; el test se enumera para auditoría de IDs/fechas.
for s in ('train','val'):
    for p in parts[s]:
        assert p.is_file(), f'Imagen ausente: {p}'
        label=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        assert label.is_file(),f'Etiqueta ausente: {label}'
assert cf.get('names') in ({0:'cerdo'},{0:'pig'},['cerdo'],['pig'])
HASHES={'data_fecha_yaml':sha256_file(YAML),'date_manifest':sha256_file(MANIFEST),
        **{s+'_txt':sha256_file(Path(cf[s])) for s in ('train','val','test')}}
PROTOCOL={"protocol_id":PROTOCOL_ID,"objective":"sensitivity of NB54: Mask R-CNN with time-based cosine LR decay under the same 60-min L4 budget",
    "only_change_vs_nb54":"learning-rate policy (constant 1e-4 -> cosine 1e-4 to 1e-6 over elapsed budget time)",
    "split_counts":{s:len(parts[s]) for s in parts},"split_dates":{s:len(dates[s]) for s in dates},
    "seeds":list(SEEDS),"gpu":GPU,"budgets_min":BUDGET_MINUTES_PER_RUN,"lr0":LR0,"lr_final_fraction":LR_FINAL_FRACTION,
    "frozen_inputs":HASHES,"torch":torch.__version__,"torchvision":torchvision.__version__,"mask_batch":MASK_BATCH,
    "test_used_for_training_or_tuning":False}
protocol_file=OUT/'protocol_config_v1.json'
if protocol_file.exists():
    prev=json.loads(protocol_file.read_text(encoding='utf-8'))
    if prev.get('frozen_inputs')!=HASHES: raise RuntimeError('Las fuentes congeladas cambiaron: ABORTAR.')
    for k in ('budgets_min','lr0','lr_final_fraction'):
        if prev.get(k)!=PROTOCOL[k]: raise RuntimeError(f'Parámetro {k} distinto al registrado; usar otra carpeta de experimento.')
else: write_json_atomic(protocol_file,PROTOCOL)
print('✅ Auditoría: train/val/test disjuntos, sin test utilizado en el ajuste. SHA YAML:',HASHES['data_fecha_yaml'])


## 4. Conjunto de datos, Mask R-CNN y evaluador COCO de validación (idénticos al NB54)

In [ ]:
import numpy as np, pandas as pd,torch,cv2,yaml,random,math
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms.functional import to_tensor

class PigMasks(Dataset):
    def __init__(self,imagelist):
        self.paths=[Path(s.strip()) for s in Path(imagelist).read_text(encoding='utf8').splitlines() if s.strip()]
        if not self.paths:raise RuntimeError('La partición está vacía: '+str(imagelist))
        missing=[str(p) for p in self.paths if not p.is_file()]
        if missing: raise FileNotFoundError('Hay imágenes ausentes: '+str(missing[:3]))
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        rgb=np.asarray(Image.open(p).convert('RGB'))
        h,w=rgb.shape[:2]
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError('Etiqueta no localizada: '+str(lab))
        masks=[];boxes=[]
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip(): continue
            vals=row.split()
            if int(vals[0])!=0:raise ValueError('Clase inesperada en '+str(lab))
            points=np.asarray([float(v) for v in vals[1:]],dtype=np.float32)
            if len(points)<6 or len(points)%2:raise ValueError('Polígono inválido en '+str(lab))
            if not np.isfinite(points).all() or np.any((points<0)|(points>1)):
                raise ValueError('Coordenadas no normalizadas en '+str(lab))
            coords=points.reshape(-1,2)*np.asarray([w,h],dtype=np.float32)
            xy=np.rint(coords).astype(np.int32)
            xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[xy],1)
            yy,xx=np.where(mask)
            if len(xx)<3: continue
            x0,x1=int(xx.min()),int(xx.max()+1);y0,y1=int(yy.min()),int(yy.max()+1)
            if x1<=x0 or y1<=y0:continue
            masks.append(mask);boxes.append([x0,y0,x1,y1])
        if not masks:raise RuntimeError('Imagen sin máscaras válidas: '+p.name)
        tar={'boxes':torch.as_tensor(boxes,dtype=torch.float32),
             'labels':torch.ones((len(masks),),dtype=torch.int64),
             'masks':torch.as_tensor(np.stack(masks),dtype=torch.uint8),
             'image_id':torch.tensor([index],dtype=torch.int64),
             'area':torch.as_tensor([int(m.sum()) for m in masks],dtype=torch.float32),
             'iscrowd':torch.zeros((len(masks),),dtype=torch.int64)}
        return to_tensor(rgb.copy()),tar

def collate_fn(batch):return tuple(zip(*batch))

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_maskrcnn(pretrained=True):
    model=maskrcnn_resnet50_fpn(weights='DEFAULT' if pretrained else None,
                               weights_backbone=None,min_size=512,max_size=768,
                               trainable_backbone_layers=3)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,2)
    in_channels=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(in_channels,256,2)
    # Congelar los mismos niveles, tanto en arranque con pesos COCO como en reanudación sin descarga.
    for name,parameter in model.backbone.body.named_parameters():
        if name.startswith(('conv1.', 'bn1.', 'layer1.')):
            parameter.requires_grad_(False)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def config_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils
from contextlib import redirect_stdout
import io

def encode_mask(arr):
    enc=maskutils.encode(np.asfortranarray(np.asarray(arr,dtype=np.uint8)))
    enc['counts']=enc['counts'].decode('ascii')
    return enc

def coco_metric_for_model(model,dataset,device,desc='Validación',include_rows=False):
    model.eval();gts=[];preds=[];image_records=[];annotations=[];aid=1
    for i in range(len(dataset)):
        image,target=dataset[i]
        height,width=image.shape[1:]
        gts.append({'id':i+1,'width':width,'height':height,'file_name':dataset.paths[i].name})
        gt_masks=target['masks'].numpy()
        for m in gt_masks:
            encoded=encode_mask(m)
            x,y,w,h=[float(v) for v in maskutils.toBbox({'size':encoded['size'],'counts':encoded['counts'].encode('ascii')})]
            annotations.append({'id':aid,'image_id':i+1,'category_id':1,'segmentation':encoded,
                                'bbox':[x,y,w,h],'area':int(m.sum()),'iscrowd':0});aid+=1
        with torch.no_grad():
            out=model([image.to(device)])[0]
        scores=out['scores'].detach().cpu().numpy()
        ms=(out['masks'].detach().cpu().numpy()[:,0]>.5) if len(scores) else np.zeros((0,height,width),dtype=bool)
        for j,score in enumerate(scores):
            if score<0.001: continue
            m=ms[j].astype(np.uint8)
            if not m.any():continue
            enc=encode_mask(m)
            preds.append({'image_id':i+1,'category_id':1,'segmentation':enc,'score':float(score)})
        if include_rows:
            filtered=np.where(scores>=.25)[0]
            predm=[ms[k].astype(bool) for k in filtered if k<len(ms)]
            p_scores=[float(scores[k]) for k in filtered if k<len(ms)]
            pairs=[]
            for gi,g in enumerate(gt_masks):
                for pj,pm in enumerate(predm):
                    inter=np.logical_and(g,pm).sum();union=np.logical_or(g,pm).sum()
                    iou=inter/max(1,union)
                    if iou>=.5:pairs.append((float(iou),gi,pj))
            used_gt=set();used_pred=set()
            for _,gi,pj in sorted(pairs,reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi);used_pred.add(pj)
            tp=len(used_gt)
            image_records.append({'imagen':dataset.paths[i].name,
                'fecha':dataset.paths[i].name[:10],
                'anotaciones_GT':len(gt_masks),'detecciones_conf_025':len(predm),
                'TP_IoU05':tp,'FP_IoU05':len(predm)-tp,'FN_IoU05':len(gt_masks)-tp,
                'confianza_media_pred':np.mean(p_scores) if p_scores else np.nan})
        if (i+1)%50==0:print(desc,i+1,'/',len(dataset))
    gt=COCO()
    gt.dataset={'images':gts,'annotations':annotations,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
        dt=gt.loadRes(preds) if preds else gt.loadRes([])
        ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    return {'seg_map50_95':float(ev.stats[0]),'seg_map50':float(ev.stats[1]),
            'seg_map75':float(ev.stats[2]),'gt':len(annotations),
            'num_images':len(dataset),'pred_coarse':len(preds)},pd.DataFrame(image_records)


## 5. Precarga de train/val en la caché de Colab (no toca el test)

In [ ]:
# Precarga (Drive -> caché de Colab). No modifica rutas, YAML ni hashes. No toca el test.
from concurrent.futures import ThreadPoolExecutor

def _leer(p):
    p = Path(p)
    lab = Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
    return len(p.read_bytes()) + len(lab.read_bytes())

t0 = time.monotonic()
for s in ('train', 'val'):
    with ThreadPoolExecutor(8) as ex:
        mb = sum(ex.map(_leer, parts[s])) / 1e6
    print(f'{s}: {len(parts[s])} imágenes precargadas ({mb:.0f} MB)')
print(f'Precarga completada en {time.monotonic()-t0:.0f} s')


## 6. Entrenamiento de Mask R-CNN con tasa de aprendizaje coseno según el tiempo transcurrido

In [ ]:
import math

def lr_at(elapsed, budget_sec):
    frac = min(max(elapsed / budget_sec, 0.0), 1.0)
    lr_min = LR0 * LR_FINAL_FRACTION
    return lr_min + 0.5 * (LR0 - lr_min) * (1 + math.cos(math.pi * frac))

def train_maskrcnn_cosine(seed):
    folder=OUT/'maskrcnn_resnet50_fpn_cosine'/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    meta=folder/'finish.json';last=folder/'maskrcnn_last.pt';best=folder/'maskrcnn_best.pt';hist=folder/'history.csv'
    budget_sec=BUDGET_MINUTES_PER_RUN*60
    if meta.exists():
        info=json.loads(meta.read_text(encoding='utf-8'))
        assert info['frozen_inputs']==HASHES and info['budget_seconds_target']==budget_sec and best.is_file()
        print('Mask R-CNN coseno',seed,'ya finalizado; no se repite.')
        return info
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    train_ds=PigMasks(cf['train']);val_ds=PigMasks(cf['val'])
    dl=DataLoader(train_ds,batch_size=MASK_BATCH,shuffle=True,num_workers=0,collate_fn=collate_fn)
    model=build_maskrcnn(pretrained=not last.exists()).to('cuda')
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR0,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    epochs=[];start_epoch=1;best_ap=-1;elapsed_prior=0.0
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        assert state['frozen_inputs']==HASHES and state['budget_seconds_target']==budget_sec and state['seed']==seed
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler']);start_epoch=state['epoch']+1
        best_ap=state['best_ap'];elapsed_prior=state['elapsed_seconds_total']
        epochs=pd.read_csv(hist).to_dict('records') if hist.exists() else []
        print('Reanudar MaskRCNN seed',seed,'desde época',start_epoch)
    launch_t0=time.monotonic();elapsed=elapsed_prior
    for epoch in range(start_epoch,MASK_EPOCH_CAP+1):
        if elapsed>=budget_sec:break
        model.train();losses=[];train_t0=time.monotonic()
        for images,targets in dl:
            images=[x.to('cuda') for x in images]
            targets=[{k:v.to('cuda') for k,v in t.items()} for t in targets]
            lr_now = lr_at(time.monotonic()-launch_t0+elapsed_prior, budget_sec)
            for g in optimizer.param_groups: g['lr'] = lr_now
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                loss_dict=model(images,targets);loss=sum(loss_dict.values())
            if not torch.isfinite(loss): raise RuntimeError('Loss Mask R-CNN no finita; inspeccionar datos/GPU')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer);scaler.update();losses.append(float(loss.item()))
            if time.monotonic()-launch_t0+elapsed_prior>=budget_sec:break
        torch.cuda.synchronize()
        time_train=time.monotonic()-train_t0
        # Validar cada dos épocas, o en la última disponible. Nunca usar test.
        time_before_val=time.monotonic()-launch_t0+elapsed_prior
        do_val=(epoch%2==0 or time_before_val>=budget_sec or epoch==MASK_EPOCH_CAP)
        val_ap=None
        if do_val:
            val_metrics,_=coco_metric_for_model(model,val_ds,torch.device('cuda'),desc=f'Val MaskRCNN {seed}/{epoch}')
            val_ap=float(val_metrics['seg_map50_95'])
            if val_ap>best_ap+1e-4:
                best_ap=val_ap
                torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            'epoch':epoch,'best_val_ap':best_ap,'seed':seed,'frozen_inputs':HASHES},best)
        elapsed=time.monotonic()-launch_t0+elapsed_prior
        row={'model':'maskrcnn_resnet50_fpn_cosine','seed':seed,'lr_end_of_epoch':optimizer.param_groups[0]['lr'],'epoch':epoch,'train_seconds':time_train,
             'elapsed_seconds_total':elapsed,'train_loss_mean':np.mean(losses),'val_AP50_95':val_ap,
             'train_images_processed_this_epoch':len(losses)*MASK_BATCH}
        epochs.append(row);pd.DataFrame(epochs).to_csv(hist,index=False)
        state={'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
               'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
               'epoch':epoch,'seed':seed,'best_ap':best_ap,
               'elapsed_seconds_total':elapsed,'budget_seconds_target':budget_sec,'frozen_inputs':HASHES}
        torch.save(state,last)
        print('MaskRCNN-coseno',seed,'epoch',epoch,'elapsed sec',round(elapsed,1),'lr',f"{optimizer.param_groups[0]['lr']:.2e}",'val_AP',val_ap)
        if elapsed>=budget_sec:break
    if not best.exists():
        # Si se interrumpió antes de la primera validación, forzar una evaluación válida.
        val_metrics,_=coco_metric_for_model(model,val_ds,torch.device('cuda'),desc='Val final')
        best_ap=float(val_metrics['seg_map50_95'])
        torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'epoch':epoch,'best_val_ap':best_ap,'seed':seed,'frozen_inputs':HASHES},best)
        elapsed=time.monotonic()-launch_t0+elapsed_prior
    info={'model':'maskrcnn_resnet50_fpn_cosine','seed':seed,'elapsed_seconds':elapsed,
          'budget_seconds_target':budget_sec,'deviation_pct':100*(elapsed-budget_sec)/budget_sec,
          'epochs_finished':len(epochs),'best_validation_seg_AP':best_ap,'best_weights_sha256':sha256_file(best),
          'frozen_inputs':HASHES,'train_dataset':'FaroPigSeg split_fecha_v1 (train/val only)',
          'architecture_recipe':'torchvision COCO v1; AdamW 1e-4 with time-based cosine decay to 1e-6; batch 1; 512-768px; backbone partial freeze',
          'test_evaluated':False,'gpu':GPU,'finished_utc':datetime.now(timezone.utc).isoformat()}
    write_json_atomic(meta,info)
    del model,optimizer,scaler,dl,train_ds,val_ds;gc.collect();torch.cuda.empty_cache()
    return info


## 7. Ejecutar las semillas pendientes (≈60 min cada una)

In [ ]:
assert GPU_NAME_REQUIRED.lower() in torch.cuda.get_device_name(0).lower()
pendientes=[s for s in SEEDS if not (OUT/'maskrcnn_resnet50_fpn_cosine'/f'seed_{s}'/'finish.json').exists()]
print('Semillas pendientes:',pendientes or 'ninguna (las 3 ya terminaron)')
for sd in pendientes:
    print('ENTRENAMIENTO Mask R-CNN coseno, semilla',sd)
    train_maskrcnn_cosine(sd)
print('✅ Entrenamiento NB55 terminado para las semillas pendientes. No se evaluó el test.')


## 8. Comparación con el NB54 (tasa constante) — solo validación

In [ ]:
rows=[]
for tag,base,label in [('maskrcnn_resnet50_fpn',OUT54/'matched','NB54 · LR constante'),('maskrcnn_resnet50_fpn_cosine',OUT,'NB55 · LR coseno')]:
    for sd in SEEDS:
        f=base/tag/f'seed_{sd}'/'finish.json'
        if f.exists():
            d=json.loads(f.read_text(encoding='utf-8'))
            assert d['frozen_inputs']==HASHES and not d['test_evaluated']
            rows.append({'receta':label,'semilla':sd,'épocas':d['epochs_finished'],'minutos':round(d['elapsed_seconds']/60,2),
                         'desviación_%':round(d['deviation_pct'],2),'mejor_AP_val':round(d['best_validation_seg_AP'],4)})
cmp=pd.DataFrame(rows)
if len(cmp):
    display(cmp)
    resumen=cmp.groupby('receta')['mejor_AP_val'].agg(['mean','std','count']).round(4)
    display(resumen)
    cmp.to_csv(OUT/'comparacion_lr_constante_vs_coseno.csv',index=False)
    pares=cmp.pivot(index='semilla',columns='receta',values='mejor_AP_val').dropna()
    if pares.shape[1]==2 and len(pares):
        dif=(pares['NB55 · LR coseno']-pares['NB54 · LR constante'])
        print('Diferencia coseno − constante por semilla:',dif.round(4).to_dict(),'| media',round(dif.mean(),4))
    plt.figure(figsize=(8,4))
    for tag,base,label,col in [('maskrcnn_resnet50_fpn',OUT54/'matched','LR constante (NB54)','#8A94A3'),('maskrcnn_resnet50_fpn_cosine',OUT,'LR coseno (NB55)','#1F3A5F')]:
        for sd in SEEDS:
            h=base/tag/f'seed_{sd}'/'history.csv'
            if h.exists():
                t=pd.read_csv(h).dropna(subset=['val_AP50_95'])
                plt.plot(t['elapsed_seconds_total']/60,t['val_AP50_95'],color=col,alpha=.85,label=label if sd==SEEDS[0] else None)
    plt.xlabel('Minutos de entrenamiento en L4');plt.ylabel('AP50:95 de máscara (validación)');plt.legend();plt.grid(alpha=.3);plt.tight_layout()
    plt.savefig(OUT/'curvas_val_lr_constante_vs_coseno.png',dpi=200);plt.show()
    print('✅ Guardado:',OUT/'comparacion_lr_constante_vs_coseno.csv')
else: print('Todavía no hay corridas terminadas.')


## 9. Cómo interpretar
- Esta comparación es **solo de validación** y sirve para decidir si la política de tasa de aprendizaje cambia la conclusión del NB54.
- Si Mask R-CNN con coseno sigue claramente por debajo de YOLO11n y YOLO26n, la diferencia **no** se explica por la tasa de aprendizaje constante.
- Si la brecha se reduce mucho, se informará y se discutirá antes del envío.
- La evaluación final en el test (una sola vez, con el evaluador COCO común del NB33 y etiquetada como retrospectiva) se hará en un notebook aparte, junto con las 9 corridas del NB54.
